# Результаты бенчмарков rangequery

На асимптотических графиках ось X — точный размер множества `data_size`, а не число команд. Для каждого размера выполняется одинаковое число запросов, поэтому графики показывают влияние размера структуры.

In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
BINARY = ROOT / 'build' / 'release' / 'src' / 'rangequery'
RESULTS = ROOT / 'benchmark-results' / 'scaling.csv'
RESULTS.parent.mkdir(exist_ok=True)
subprocess.run([sys.executable, str(ROOT / 'tools' / 'benchmark.py'), '--binary', str(BINARY), '--output', str(RESULTS), '--data-sizes', '1000', '3000', '10000', '--query-count', '10000', '--repetitions', '3', '--seed', '42'], check=True)
results = pd.read_csv(RESULTS)
results.head()

In [ ]:
summary = results.groupby(['structure', 'data_size'], as_index=False)['ns_per_query'].median()
for structure, group in summary.groupby('structure'):
    plt.loglog(group['data_size'], group['ns_per_query'], marker='o', label=structure)
plt.xlabel('размер множества'); plt.ylabel('медианное время на запрос (нс)'); plt.title('Масштабирование структур')
plt.grid(True, which='both', alpha=.25); plt.legend(); plt.show()

In [ ]:
# Нормирование показывает форму асимптотики, а не абсолютные константы.
for structure, group in summary.groupby('structure'):
    n = group['data_size']
    normalized = group['ns_per_query'] / group['ns_per_query'].iloc[0]
    plt.loglog(n, normalized, marker='o', label=structure)
n = summary['data_size'].drop_duplicates().sort_values()
plt.loglog(n, n / n.iloc[0], '--', label='O(n)')
plt.loglog(n, (n * n.apply(lambda value: __import__('math').log2(value))) / (n.iloc[0] * __import__('math').log2(n.iloc[0])), '--', label='O(n log n)')
plt.loglog(n, n.apply(lambda value: __import__('math').log2(value)) / __import__('math').log2(n.iloc[0]), '--', label='O(log n)')
plt.xlabel('размер множества'); plt.ylabel('нормированное время на запрос'); plt.title('Эталонные асимптотические кривые')
plt.grid(True, which='both', alpha=.25); plt.legend(); plt.show()

## Локальность обращений

Здесь выполняются 100 000 rank-запросов на множествах среднего размера. `uniform` обращается ко всему диапазону ключей, а `hotspot` концентрирует обращения вокруг центральных 5 % ключей.

In [ ]:
LOCALITY_SIZES = ('1000', '3000', '10000')
LOCALITY_QUERY_COUNT = '100000'
locality_results_path = ROOT / 'benchmark-results' / 'locality.csv'
subprocess.run([sys.executable, str(ROOT / 'tools' / 'benchmark.py'), '--binary', str(BINARY), '--output', str(locality_results_path), '--data-sizes', *LOCALITY_SIZES, '--query-count', LOCALITY_QUERY_COUNT, '--query-kind', 'rank', '--localities', 'uniform', 'hotspot', '--repetitions', '3', '--seed', '42'], check=True)
locality_results = pd.read_csv(locality_results_path)
locality_summary = locality_results.groupby(['structure', 'locality', 'data_size'], as_index=False)['ns_per_query'].median()
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for axis, (structure, group) in zip(axes, locality_summary.groupby('structure')):
    for locality, series in group.groupby('locality'):
        axis.loglog(series['data_size'], series['ns_per_query'], marker='o', label=locality)
    axis.set_title(structure); axis.set_xlabel('размер множества'); axis.grid(True, which='both', alpha=.25)
axes[0].set_ylabel('медианное время на запрос (нс)'); axes[-1].legend(title='локальность')
fig.suptitle('Влияние локальности обращений: 100 000 rank-запросов'); fig.tight_layout(); plt.show()